# Group 2: images → camera poses → Gaussian Splat

Runtime → **Change runtime type → T4 GPU** (or better) before running anything.

Every step is a script in `group2/`; this notebook only calls them in order. Run sections 1–2 **before the
meeting** (installing and compiling takes ~5–10 min); sections 3–6 are the live run.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torch CUDA", torch.version.cuda,
      "| GPU available:", torch.cuda.is_available())
!nvcc --version | tail -n 2

## 2. Get the code and install

`gsplat` has no prebuilt wheel for Colab's current Python/torch, so it compiles its CUDA kernels the
first time it runs (a few minutes). The last cell here triggers that compile now, not during the meeting.
The torch CUDA version and `nvcc` version printed above should share a major version (both 12.x or both 13.x).

The repo is public, so this clones with no setup. If it's ever made private, create a fine-grained
GitHub token with read-only **Contents** access to this one repo, add it in Colab under the key icon
(Secrets) as `GITHUB_TOKEN`, and allow this notebook to read it. Never paste the token into a cell.

In [ ]:
REPO = "github.com/arjunch1001/acmAI-3d-reconstruction.git"
BRANCH = "group2-pipeline"  # switch to "main" once it's merged
from google.colab import userdata
try:
    auth = userdata.get("GITHUB_TOKEN") + "@"
except Exception:
    auth = ""  # no secret set: fine while the repo is public
!rm -rf /content/repo && git clone -q --branch $BRANCH https://{auth}{REPO} /content/repo
%cd /content/repo
!git remote set-url origin https://{REPO}  # don't leave the token in .git/config
!ls group2

In [ ]:
# pycolmap-cuda12 runs SIFT on the GPU; fall back to the CPU build if it can't load here.
!pip install -q pycolmap-cuda12 gsplat ninja
import importlib, subprocess
try:
    import pycolmap
    assert pycolmap.has_cuda
except Exception as e:
    print("GPU pycolmap unavailable, using CPU build:", e)
    subprocess.run("pip uninstall -y -q pycolmap-cuda12 && pip install -q pycolmap", shell=True)
    import pycolmap; importlib.reload(pycolmap)
print("pycolmap", pycolmap.__version__, "| CUDA:", pycolmap.has_cuda)

In [ ]:
# Compile gsplat's CUDA kernels now (cached until the runtime is recycled).
import time, torch
from gsplat import rasterization
t = time.time()
n = 1000
out, _, _ = rasterization(torch.randn(n, 3, device="cuda") + torch.tensor([0, 0, 5.0], device="cuda"),
                          torch.nn.functional.normalize(torch.randn(n, 4, device="cuda"), dim=-1),
                          torch.full((n, 3), 0.05, device="cuda"), torch.full((n,), 0.8, device="cuda"),
                          torch.rand(n, 3, device="cuda"), torch.eye(4, device="cuda")[None],
                          torch.tensor([[[300., 0, 160], [0, 300., 120], [0, 0, 1]]], device="cuda"), 320, 240)
print(f"gsplat ready ({time.time() - t:.0f}s incl. compile), test render {tuple(out.shape)}")

## 3. Get Group 1's frames

Upload their zip (or put it in Drive and set `ZIP` to its path). If you already ran SfM on a laptop,
upload that `sfm` folder's `dataset/` zipped instead and skip to section 5.

In [ ]:
ZIP = ""  # e.g. "/content/drive/MyDrive/acm/frames.zip"; leave empty to upload
if not ZIP:
    from google.colab import files
    ZIP = "/content/" + next(iter(files.upload()))
!rm -rf /content/frames && mkdir -p /content/frames && unzip -q -j "$ZIP" -d /content/frames
import glob
n = len(glob.glob("/content/frames/*.jp*g") + glob.glob("/content/frames/*.png"))
assert n, f"No .jpg/.png images came out of {ZIP}. Check that it's the right zip."
print(n, "frames in /content/frames")

In [ ]:
# Errors (unreadable or mixed-size images) stop here; warnings are for you to judge. Also writes 1280px copies.
!python group2/check_images.py /content/frames --resize-to /content/frames_1280
assert _exit_code == 0, "check_images.py found errors (listed above). Fix them before SfM."
# A notebook "!" command never stops the run on failure, so each step checks _exit_code (the last command's exit status).

## 4. Camera poses (SfM)

In [ ]:
import os
assert os.path.isdir("/content/frames_1280"), "Run both cells in section 3 first."
# pipefail: report run_sfm.py's exit status, not grep's (grep only hides COLMAP's log lines).
!bash -c 'set -o pipefail; python group2/run_sfm.py /content/frames_1280 /content/sfm --overwrite 2>&1 | grep -vE "^[IW][0-9]{8}"'
assert _exit_code == 0, "run_sfm.py failed (see above)."
!python group2/sfm_report.py /content/sfm /content/frames_1280
from IPython.display import Image, display
if os.path.exists("/content/sfm/sfm_report.png"):
    display(Image("/content/sfm/sfm_report.png"))
assert _exit_code == 0, "sfm_report.py says don't train on this model (see ERROR lines above)." 

## 5. Train

`--steps 3000` is the spec's fast run. Watch the loss fall and the Gaussian count grow until 70% of the
steps, then hold. Previews (photo | render) land in `/content/run/previews/`.
For the dry run, add `--holdout 8` to also measure PSNR on views the model never trained on.

In [ ]:
!python group2/train.py /content/sfm/dataset /content/run --steps 3000
assert _exit_code == 0, "train.py failed (see above)." 

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, glob
m = pd.read_csv("/content/run/metrics.csv")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].plot(m.step, m.loss); ax[0].set_title("loss")
ax[1].plot(m.step, m.psnr); ax[1].set_title("train PSNR (dB)")
ax[2].plot(m.step, m.gaussians); ax[2].set_title("Gaussians")
plt.show()
display(Image(sorted(glob.glob("/content/run/previews/*.png"))[-1]))
print(open("/content/run/summary.json").read())

## 6. Hand off to Group 3

In [ ]:
from google.colab import files
files.download("/content/run/splat.ply")